In [85]:
import requests
import pandas as pd
from io import StringIO

url = "https://raw.githubusercontent.com/MohammedHameds/test1455/refs/heads/main/products.csv"

response = requests.get(url)

products = pd.read_csv(StringIO(response.text))

products.to_csv('products.csv', index=False)

### Customers

In [86]:
customers = pd.read_csv('customers.csv')
customers

,customer_id,first_name,last_name,city,age,email
0,1,Ahmed,Hassan,Cairo,25,ahmed@email.com
1,2,Sara,Ali,Giza,31,sara@email.com
2,3,Omar,Mohamed,cairo,17,omar@email.com
3,4,Mona,Ibrahim,Alexandria,28,NaN
4,5,Ahmed,Hassan,Cairo,25,ahmed@email.com


In [87]:
customers = customers.drop_duplicates(subset=["first_name", "last_name", "email"])
customers.duplicated()

0    False
1    False
2    False
3    False
dtype: bool

In [88]:
customers['city'] = customers['city'].str.strip().str.title()

C:\Users\zeina\AppData\Local\Temp\ipykernel_33160\2591318126.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  customers['city'] = customers['city'].str.strip().str.title()


In [89]:
customers["age"] = pd.to_numeric(customers["age"], errors="coerce")
customers = customers[(customers["age"] >= 5) & (customers["age"] <= 100)]

C:\Users\zeina\AppData\Local\Temp\ipykernel_33160\3124957800.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  customers["age"] = pd.to_numeric(customers["age"], errors="coerce")


In [90]:
customers["email"] = customers["email"].fillna("unknown@email.com")
customers["email"].isna().sum()


np.int64(0)

In [91]:
customers['full_name'] = customers['first_name'] + ' ' + customers['last_name']
customers

,customer_id,first_name,last_name,city,age,email,full_name
0,1,Ahmed,Hassan,Cairo,25,ahmed@email.com,Ahmed Hassan
1,2,Sara,Ali,Giza,31,sara@email.com,Sara Ali
2,3,Omar,Mohamed,Cairo,17,omar@email.com,Omar Mohamed
3,4,Mona,Ibrahim,Alexandria,28,unknown@email.com,Mona Ibrahim


In [92]:
customers.to_csv("clean_customers.csv", index=False)

### Orders

In [93]:
orders = pd.read_parquet("orders.parquet")
orders

,order_id,customer_id,product_id,quantity,order_date
0,1001,1,101,2,2026-09-01
1,1002,2,102,1,2026-09-01
2,1003,3,103,3,2026-09-02
3,1004,4,101,1,2026-09-02
4,1005,1,104,2,2026-09-03
5,1006,5,102,1,2026-09-03
6,1007,2,105,4,2026-09-04
7,1008,3,101,1,2026-09-04
8,1009,4,103,2,2026-09-05
9,1010,1,104,1,2026-09-05


In [94]:
orders['order_id'].duplicated().sum()

np.int64(1)

In [95]:
orders = orders.drop_duplicates(subset="order_id")
orders['order_id'].duplicated().sum()

np.int64(0)

In [96]:
orders["quantity"] = pd.to_numeric(orders["quantity"], errors="coerce")
orders = orders.dropna(subset=["quantity"])
orders = orders[orders["quantity"] > 0]

C:\Users\zeina\AppData\Local\Temp\ipykernel_33160\4225503788.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  orders["quantity"] = pd.to_numeric(orders["quantity"], errors="coerce")


In [97]:
orders["order_date"] = pd.to_datetime(orders["order_date"])

In [98]:
orders["customer_id"] = orders["customer_id"].replace({5: 1})
orders = orders[orders["customer_id"].isin(customers["customer_id"])]

In [99]:
orders.to_csv("cleaned_orders.csv", index=False)

### Products

In [100]:
products = pd.read_csv('products.csv')
products

,product_id,product_name,category,unit_price
0,101,Laptop,Electronics,25000
1,102,Mouse,Accessories,500
2,103,Keyboard,Accessories,1200
3,104,Monitor,Electronics,7000
4,105,Headphones,Accessories,2500
5,106,Webcam,Accessories,1800


In [101]:
products = products.drop_duplicates()

In [102]:
products["product_name"] = products["product_name"].str.strip().str.title()
products["category"] = products["category"].str.strip().str.title()

In [103]:
products['unit_price'] = pd.to_numeric(products['unit_price'], errors= "coerce")
products = products.dropna(subset=["unit_price"])
products = products[products["unit_price"] > 0]

In [104]:
products.to_csv('cleaned_products.csv', index=False)

### Mergeing

In [105]:
sales = orders.merge(customers, on="customer_id", how="inner")
sales = sales.merge(products, on="product_id", how="inner")

sales

,order_id,customer_id,product_id,quantity,order_date,first_name,last_name,city,age,email,full_name,product_name,category,unit_price
0,1001,1,101,2,2026-09-01,Ahmed,Hassan,Cairo,25,ahmed@email.com,Ahmed Hassan,Laptop,Electronics,25000
1,1002,2,102,1,2026-09-01,Sara,Ali,Giza,31,sara@email.com,Sara Ali,Mouse,Accessories,500
2,1003,3,103,3,2026-09-02,Omar,Mohamed,Cairo,17,omar@email.com,Omar Mohamed,Keyboard,Accessories,1200
3,1004,4,101,1,2026-09-02,Mona,Ibrahim,Alexandria,28,unknown@email.com,Mona Ibrahim,Laptop,Electronics,25000
4,1005,1,104,2,2026-09-03,Ahmed,Hassan,Cairo,25,ahmed@email.com,Ahmed Hassan,Monitor,Electronics,7000
5,1006,1,102,1,2026-09-03,Ahmed,Hassan,Cairo,25,ahmed@email.com,Ahmed Hassan,Mouse,Accessories,500
6,1007,2,105,4,2026-09-04,Sara,Ali,Giza,31,sara@email.com,Sara Ali,Headphones,Accessories,2500
7,1008,3,101,1,2026-09-04,Omar,Mohamed,Cairo,17,omar@email.com,Omar Mohamed,Laptop,Electronics,25000
8,1009,4,103,2,2026-09-05,Mona,Ibrahim,Alexandria,28,unknown@email.com,Mona Ibrahim,Keyboard,Accessories,1200
9,1010,1,104,1,2026-09-05,Ahmed,Hassan,Cairo,25,ahmed@email.com,Ahmed Hassan,Monitor,Electronics,7000


In [106]:
sales["total_amount"] = sales["quantity"] * sales["unit_price"]

sales

,order_id,customer_id,product_id,quantity,order_date,first_name,last_name,city,age,email,full_name,product_name,category,unit_price,total_amount
0,1001,1,101,2,2026-09-01,Ahmed,Hassan,Cairo,25,ahmed@email.com,Ahmed Hassan,Laptop,Electronics,25000,50000
1,1002,2,102,1,2026-09-01,Sara,Ali,Giza,31,sara@email.com,Sara Ali,Mouse,Accessories,500,500
2,1003,3,103,3,2026-09-02,Omar,Mohamed,Cairo,17,omar@email.com,Omar Mohamed,Keyboard,Accessories,1200,3600
3,1004,4,101,1,2026-09-02,Mona,Ibrahim,Alexandria,28,unknown@email.com,Mona Ibrahim,Laptop,Electronics,25000,25000
4,1005,1,104,2,2026-09-03,Ahmed,Hassan,Cairo,25,ahmed@email.com,Ahmed Hassan,Monitor,Electronics,7000,14000
5,1006,1,102,1,2026-09-03,Ahmed,Hassan,Cairo,25,ahmed@email.com,Ahmed Hassan,Mouse,Accessories,500,500
6,1007,2,105,4,2026-09-04,Sara,Ali,Giza,31,sara@email.com,Sara Ali,Headphones,Accessories,2500,10000
7,1008,3,101,1,2026-09-04,Omar,Mohamed,Cairo,17,omar@email.com,Omar Mohamed,Laptop,Electronics,25000,25000
8,1009,4,103,2,2026-09-05,Mona,Ibrahim,Alexandria,28,unknown@email.com,Mona Ibrahim,Keyboard,Accessories,1200,2400
9,1010,1,104,1,2026-09-05,Ahmed,Hassan,Cairo,25,ahmed@email.com,Ahmed Hassan,Monitor,Electronics,7000,7000


In [107]:
sales = sales.rename(columns={"full_name": "customer", "product_name": "product"})
sales = sales[["order_id", "customer", "product", "quantity", "unit_price", "total_amount"]]

In [108]:
sales.to_csv("sales.csv", index=False)